# Phase 6: Collaborative Filtering (ItemKNN, BiasedMF, ImplicitALS)

This notebook implements, tunes, and evaluates three foundational collaborative filtering paradigms for CineMatch AI:
1. **ItemKNN**: Memory-based neighborhood collaborative filtering using mean-centered cosine similarity.
2. **Biased Matrix Factorization (BiasedMF)**: Pointwise explicit rating estimation using global, user, and item biases trained via SGD.
3. **Implicit Alternating Least Squares (ImplicitALS)**: Pairwise preference ranking over positive interactions ($r \ge 4.0$).

### Core Objectives:
- Implement models conforming to the common `Recommender` interface.
- Benchmark validation performance ($K=10$, Precision, Recall, NDCG, MAP, HitRate, Coverage, RMSE/MAE, Latency).
- Rigorously investigate: **Why does low rating-prediction RMSE not guarantee high top-K ranking performance?**


## 1. Setup & Data Ingestion

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.collaborative_model import BiasedMF, ImplicitALS, ItemKNN
from ml.evaluate import evaluate_model

DATA_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
ARTIFACTS_DIR = Path("../artifacts")

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
movies_df = pd.read_parquet(DATA_DIR / "movies_cleaned.parquet")
catalog_size = len(movies_df)

print(f"Train interactions : {len(train_df):,}")
print(f"Val interactions   : {len(val_df):,}")
print(f"Total catalog items: {catalog_size:,}")

Train interactions : 80,650
Val interactions   : 10,093
Total catalog items: 9,742


E:\CineMatch-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. ItemKNN: Neighborhood Collaborative Filtering

In [2]:
# Instantiate and fit ItemKNN with tuned top-10 neighbors
item_knn = ItemKNN(n_neighbors=10, min_similarity=0.0)
item_knn.fit(train_df)

knn_metrics = evaluate_model(
    item_knn, val_df, train_df=train_df, k=10, total_catalog_size=catalog_size
)
print("--- ItemKNN Validation Benchmark (K=10) ---")
for k_m, val in knn_metrics.items():
    if isinstance(val, float):
        print(f"{k_m:<20}: {val:.4f}")
    else:
        print(f"{k_m:<20}: {val}")

--- ItemKNN Validation Benchmark (K=10) ---
precision@10        : 0.0209
recall@10           : 0.0447
map@10              : 0.0223
ndcg@10             : 0.0413
hit_rate@10         : 0.1699
catalog_coverage    : 0.1591
rmse                : 0.9721
mae                 : 0.7354
avg_latency_ms      : 0.7872
evaluated_users     : 610
evaluated_relevant_users: 565


## 3. Explicit Matrix Factorization (BiasedMF via SGD)

In [3]:
# Instantiate and train BiasedMF with tuned hyperparameters
biased_mf = BiasedMF(n_factors=16, reg=0.02, lr=0.005, n_epochs=15, seed=42)
biased_mf.fit(train_df)

mf_metrics = evaluate_model(
    biased_mf, val_df, train_df=train_df, k=10, total_catalog_size=catalog_size
)
print("--- BiasedMF Validation Benchmark (K=10) ---")
for k_m, val in mf_metrics.items():
    if isinstance(val, float):
        print(f"{k_m:<20}: {val:.4f}")
    else:
        print(f"{k_m:<20}: {val}")

--- BiasedMF Validation Benchmark (K=10) ---
precision@10        : 0.0198
recall@10           : 0.0271
map@10              : 0.0102
ndcg@10             : 0.0256
hit_rate@10         : 0.1575
catalog_coverage    : 0.0059
rmse                : 0.8950
mae                 : 0.6881
avg_latency_ms      : 0.1595
evaluated_users     : 610
evaluated_relevant_users: 565


## 4. Implicit Alternating Least Squares (ImplicitALS)

In [4]:
# Instantiate and train ImplicitALS (factors=64, reg=0.1)
implicit_als = ImplicitALS(n_factors=64, regularization=0.10, iterations=15, seed=42)
implicit_als.fit(train_df)

als_metrics = evaluate_model(
    implicit_als, val_df, train_df=train_df, k=10, total_catalog_size=catalog_size
)
print("--- ImplicitALS Validation Benchmark (K=10) ---")
for k_m, val in als_metrics.items():
    if isinstance(val, float):
        print(f"{k_m:<20}: {val:.4f}")
    else:
        print(f"{k_m:<20}: {val}")

E:\CineMatch-AI\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


--- ImplicitALS Validation Benchmark (K=10) ---
precision@10        : 0.0458
recall@10           : 0.1093
map@10              : 0.0498
ndcg@10             : 0.0890
hit_rate@10         : 0.3363
catalog_coverage    : 0.0707
rmse                : None
mae                 : None
avg_latency_ms      : 0.4954
evaluated_users     : 610
evaluated_relevant_users: 565


## 5. Hyperparameter Tuning Analysis

In [5]:
tuning_csv = REPORTS_DIR / "tuning_log.csv"
if tuning_csv.exists():
    tuning_df = pd.read_csv(tuning_csv)
    print("Hyperparameter Search Log:")
    display(
        tuning_df[
            [
                "model",
                "hyperparameters",
                "ndcg@k",
                "recall@k",
                "hit_rate@k",
                "rmse",
                "catalog_coverage",
            ]
        ]
    )

Hyperparameter Search Log:


,model,hyperparameters,ndcg@k,recall@k,hit_rate@k,rmse,catalog_coverage
0,ItemKNN,n_neighbors=10; min_similarity=0.0,0.0413,0.0447,0.1699,0.9721,0.1591
1,ItemKNN,n_neighbors=20; min_similarity=0.0,0.0274,0.0297,0.1150,0.9768,0.2016
2,ItemKNN,n_neighbors=40; min_similarity=0.0,0.0127,0.0145,0.0708,0.9797,0.2459
3,ItemKNN,n_neighbors=20; min_similarity=0.1,0.0261,0.0268,0.1044,0.9762,0.2021
4,BiasedMF,n_factors=16; reg=0.02; lr=0.005; n_epochs=15,0.0256,0.0271,0.1575,0.8950,0.0059
5,BiasedMF,n_factors=16; reg=0.05; lr=0.005; n_epochs=15,0.0240,0.0257,0.1540,0.8959,0.0054
6,BiasedMF,n_factors=32; reg=0.02; lr=0.005; n_epochs=15,0.0265,0.0268,0.1540,0.8957,0.0065
7,BiasedMF,n_factors=32; reg=0.05; lr=0.005; n_epochs=15,0.0243,0.0263,0.1540,0.8968,0.0053
8,BiasedMF,n_factors=32; reg=0.1; lr=0.005; n_epochs=15,0.0230,0.0239,0.1504,0.8986,0.0052
9,ImplicitALS,n_factors=16; regularization=0.01; iterations=15,0.0876,0.1013,0.3451,NaN,0.0489


## 6. Comprehensive Validation Comparison Across All Models

In [6]:
results_csv = REPORTS_DIR / "results_val.csv"
if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    cols = [
        c
        for c in [
            "model",
            "k",
            "precision@k",
            "recall@k",
            "map@k",
            "ndcg@k",
            "hit_rate@k",
            "rmse",
            "catalog_coverage",
            "avg_latency_ms",
            "notes",
        ]
        if c in results_df.columns
    ]
    display(results_df[cols])

,model,k,precision@k,recall@k,map@k,ndcg@k,hit_rate@k,catalog_coverage,avg_latency_ms,notes
0,RandomRecommender,10,0.0012,0.0009,0.0006,0.0016,0.0124,0.4661,0.55,Sanity floor (random sample)
1,PopularityRecommender,10,0.0333,0.0585,0.0279,0.0550,0.2212,0.0101,0.01,Ranked by train rating count
2,WeightedRatingRecommender,10,0.0340,0.0574,0.0277,0.0549,0.2195,0.0074,0.01,Bayesian average (m=100.0 tuned on val)
3,ContentRecommender,10,0.0023,0.0037,0.0018,0.0038,0.0195,0.3381,0.28,Genres (1.0) + Tags TF-IDF (1.2) + Year decade...
4,ItemKNN,10,0.0209,0.0447,0.0223,0.0413,0.1699,0.1591,0.78,"Mean-centered cosine ({'n_neighbors': 10, 'min..."
5,BiasedMF,10,0.0198,0.0271,0.0102,0.0256,0.1575,0.0059,0.14,"Explicit SGD ({'n_factors': 16, 'reg': 0.02, '..."
6,ImplicitALS,10,0.0458,0.1093,0.0498,0.0890,0.3363,0.0707,0.51,"ALS on ratings>=4 ({'n_factors': 64, 'regulari..."


## 7. Analysis: Optimizing RMSE vs. Optimizing Top-K Ranking

A classic paradox in recommendation systems is that **minimizing rating-prediction error (RMSE) does not necessarily yield superior top-K recommendation quality (NDCG / Recall / Hit Rate)**. Our empirical numbers demonstrate this divergence clearly:

| Model Paradigm | Objective | Validation RMSE | NDCG@10 | Recall@10 | HitRate@10 | Catalog Coverage |
|---|---|---|---|---|---|---|
| **BiasedMF** | Explicit rating error (pointwise L2) | **0.8950** | **0.0256** | **0.0271** | **0.1575** | 0.0059 |
| **ImplicitALS** | Pairwise preference (implicit feedback) | *N/A* | **0.0890** (+248%) | **0.1093** (+303%) | **0.3363** (+114%) | 0.0707 |
| **Popularity** | Interaction frequency | *N/A* | **0.0550** | **0.0585** | **0.2212** | 0.0101 |

### Key Reasons for this Divergence:
1. **The Asymmetry of User Needs:**
   - RMSE treats all errors equally: mispredicting a 1.5-star movie as 2.0 has the exact same loss impact as mispredicting a 4.5-star favorite as 4.0.
   - However, recommendation users only consume the top 10 items. Whether a bad movie is predicted as 1.5 or 2.5 is completely irrelevant to the user experience; what matters is whether the user's favorite items reach positions 1 through 10.
2. **Bias Dominance in Explicit Factorization:**
   - In BiasedMF, the prediction is dominated by $\mu + b_u + b_i$. Because 10% of movies account for 60% of all ratings with high average grades, positive item biases $b_i$ push familiar items to the top, while slight errors on thousands of unrated candidate items introduce rank inversions.
3. **Explicit vs. Implicit Feedback Formulation:**
   - BiasedMF only observes rated pairs $(u, i)$ and assumes missing entries are "missing at random".
   - ImplicitALS treats unrated movies as negative/background signal with low confidence and explicitly trains latent representations to distinguish positive interactions ($r \ge 4.0$) from the background.
   - Consequently, ImplicitALS achieves an **NDCG@10 of 0.0890** and **HitRate@10 of 33.6%** (surpassing both Popularity and explicit BiasedMF), even though it cannot compute explicit rating RMSE.
